In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [3]:
from src.models.Autoencoder import ResNet50AutoEncoder

pretrained_model = ResNet50AutoEncoder()

In [4]:
print(pretrained_model.model.encoder)

ResNetEncoder(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (conv2): EncoderBottleneckBlock(
    (00 MaxPooling): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (01 EncoderLayer): EncoderBottleneckLayer(
      (weight_layer1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (weight_layer2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (weight_layer3): Sequential(
        (0): Conv2d(64, 256, kernel_size

In [3]:
import torch
import torch.nn as nn

class CIFAR10Upscaler(nn.Module):
    def __init__(self):
        super(CIFAR10Upscaler, self).__init__()
        self.conv1 = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1)  # Keep 3 channels
        self.conv2 = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)  # 32x32 -> 64x64
        self.upsample2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)  # 64x64 -> 128x128
        self.upsample3 = nn.Upsample(scale_factor=1.75, mode='bilinear', align_corners=True)  # 128x128 -> 224x224
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.upsample1(x)
        x = self.relu(self.conv2(x))
        x = self.upsample2(x)
        x = self.relu(self.conv3(x))
        x = self.upsample3(x)
        return x


In [4]:
import torch
import torch.nn as nn

class CIFAR10Downscaler(nn.Module):
    def __init__(self):
        super(CIFAR10Downscaler, self).__init__()
        # Convolutional layers to downscale dimensions while keeping 3 channels
        self.conv1 = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1)
        
        # Pooling layers to downscale spatial dimensions
        self.pool1 = nn.AdaptiveAvgPool2d((128, 128))  # 224x224 -> 128x128
        self.pool2 = nn.AdaptiveAvgPool2d((64, 64))    # 128x128 -> 64x64
        self.pool3 = nn.AdaptiveAvgPool2d((32, 32))    # 64x64 -> 32x32
        
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.relu(self.conv3(x))
        x = self.pool3(x)
        return x


In [5]:
class CombindedModel(nn.Module):
    def __init__(self, pretrained_model):
        super(CombindedModel, self).__init__()
        self.upscaler = CIFAR10Upscaler()
        self.pretrained_model = pretrained_model
        
        for param in self.pretrained_model.parameters():
            param.requires_grad = False
        
        self.downscaler = CIFAR10Downscaler()
        
    def forward(self, x):
        x = self.upscaler(x)
        x = self.pretrained_model(x)
        x = self.downscaler(x)
        return x

In [6]:
dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transforms.ToTensor())
cats_dataset = [(img, label) for (img, label) in dataset if label == 3]
cats_dataloader = torch.utils.data.DataLoader(cats_dataset, batch_size=32, shuffle=True)

Files already downloaded and verified


In [7]:
from torch import optim

combined_model = CombindedModel(pretrained_model)
optimizer = optim.Adam([param for param in combined_model.parameters() if param.requires_grad], lr=0.001)
criterion = nn.MSELoss()

for epoch in range(5):  # Few epochs to adapt the preprocessor layer
    for images, _ in cats_dataloader:
        optimizer.zero_grad()
        outputs = combined_model(images)
        loss = criterion(outputs, images)
        print("Loss: ", loss.item())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Loss:  0.23649907112121582
Loss:  0.230322003364563
Loss:  0.19789282977581024
Loss:  0.2236623913049698
Loss:  0.22923021018505096
Loss:  0.24018383026123047
Loss:  0.2199563980102539
Loss:  0.17616121470928192
Loss:  0.22374622523784637
Loss:  0.21840046346187592
Loss:  0.17238043248653412
Loss:  0.1871916502714157
Loss:  0.1827249974012375
Loss:  0.1633366048336029
Loss:  0.17303162813186646
Loss:  0.1721441000699997
Loss:  0.16178523004055023
Loss:  0.1581202745437622
Loss:  0.1770903319120407
Loss:  0.1384362131357193
Loss:  0.14003239572048187
Loss:  0.15004770457744598
Loss:  0.14109693467617035
Loss:  0.12179090827703476
Loss:  0.13430188596248627
Loss:  0.14157207310199738
Loss:  0.1251995712518692
Loss:  0.12386754900217056
Loss:  0.1054549589753151
Loss:  0.11021434515714645
Loss:  0.0967138409614563
Loss:  0.12310656905174255
Loss:  0.1036619320511818
Loss:  0.1109650507569313
Loss:  0.07982105761766434
Loss:  0.06520220637321472
Loss:  0.08397262543439865
Loss:  0.06664792

KeyboardInterrupt: 